# Official vs street: 90 days

The street rate of a currency from the **EMpulse Street FX Index** next to the official rate, over the last 90 days, with the premium.
The `official-vs-street` product is part of the Pro plan (https://empulse.ai/pricing); the street series alone works with a free key (https://empulse.ai/#key).

Rates are local currency per 1 USDT (counted as 1 US dollar); a rising number means the currency weakened. Not investment advice.

In [ ]:
import os
import pandas as pd
import requests

API = "https://api.empulse.ai"
HEADERS = {"X-API-Key": os.environ["EMPULSE_API_KEY"]}
CUR = "EGP"

def get(path, **params):
    r = requests.get(f"{API}{path}", headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

street = get(f"/v1/rates/{CUR}")                       # the last 90 days
street["day"] = pd.to_datetime(street["day"])
street.tail()

In [ ]:
premium = get(f"/v1/official-vs-street/{CUR}/history")   # Pro: official rate, street mid, premium per day
premium["day"] = pd.to_datetime(premium["day"])
both = street.merge(premium[["day", "official", "premium_pct"]], on="day", how="left")
both[["day", "mid", "official", "premium_pct", "confidence"]].tail(10)

In [ ]:
ax = both.plot(x="day", y=["mid", "official"], figsize=(10, 4), title=f"{CUR} per USD: street (EMpulse Street FX Index) vs official, 90 days")
ax.set_ylabel(f"{CUR} per 1 USD")
ax.figure.text(0.01, 0.01, "Source: EMpulse Street FX Index, https://empulse.ai/rates/" + CUR, fontsize=8)